<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Summaries and follow-ups

**[Summaries and follow-ups](https://learning.nextia-ai.com/courses/speech/m06/summaries-and-follow-ups/)** · Speech AI: Transcribe, Understand and Speak · Module 6, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** check an LLM's call record (reason, summary, promise, next step, recording notice) in three ways: its form, each field against the labels, and the promise against Larkfield's order system. You will see how one misheard digit stops a check.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: the LLM answers are recorded. |
| **You should know** | The call reasons and the recorded LLM answers, from [Why did they call?](https://learning.nextia-ai.com/courses/speech/m06/why-did-they-call/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m06/summaries-and-follow-ups/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M06-L02-summaries-and-follow-ups/summaries-and-follow-ups-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 3 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "0ab1c11490f1e0341f11f9cf0fc061abde616a9297fbc830cde001ef60daa01a",
    "data/orders.json": "74f313e9e6c6eac155c655f2c4ed5016ae8aa6076c38ca88edf7c09ed901aeda",
    "data/recordings/asr/whisper-small.jsonl": "67f27c0e6f5aa73d0e67cdb4b69c0e2d22c5f03a5dd74be14cbd8db0df17cb3f",
    "data/recordings/llm/reason-reference.jsonl": "bef65a1ea9166d96941071bc80ba858d5e9b21c81a86a8be707f61e026ebc282",
    "data/recordings/llm/reason-whisper-small.jsonl": "82734affbfebd7d77900e0dfa126f5eaa951a011953c2e14690decdaeaf7b932",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The call records

Azure OpenAI `chat-small` read every validation and test call once and filled in a **call record**: the reason, a short summary, the promise the agent made, the next step, whether the agent said the recording notice, and whether somebody must act after the call. The answers were recorded, so you need no key. There are two sets: one from the exact transcripts, one from Whisper small's transcripts.

In [ ]:
calls = load_calls()
validation = [c for c in calls.values() if c["split"] == "validation"]
test = [c for c in calls.values() if c["split"] == "test"]
orders = {o["order"]: o for o in json.load(open("data/orders.json", encoding="utf-8"))}

def load_records(name):
    """The recorded call records: one LLM answer (JSON) for each validation and test call."""
    rows = [json.loads(line) for line in open(f"data/recordings/llm/{name}.jsonl", encoding="utf-8") if line.strip()]
    return {r["call_id"]: r["answer"] for r in rows}

records = load_records("reason-reference")          # the LLM read the exact transcript
records_asr = load_records("reason-whisper-small")  # the LLM read Whisper small's transcript
recognised = {}
for line in open("data/recordings/asr/whisper-small.jsonl", encoding="utf-8"):
    row = json.loads(line)
    if row["kind"] == "phone":
        recognised[row["call_id"]] = row["text"]
print(len(validation), "validation calls,", len(test), "test calls,", len(records), "records,", len(orders), "orders")

> **Check.** 50 validation calls, 60 test calls, 110 records in each set, 160 orders in the fake order system.

## 2. One call and its record

Read call C-0161 and its record. Every call is synthetic: open text-to-speech voices read scripts that we wrote.

> **Predict.** Is the promise in the record right? Read what the agent says just before it.

In [ ]:
call = calls["C-0161"]
for turn in call["turns"]:
    print(f"{turn['speaker']:6} {turn['text']}")
print()
print(json.dumps(records["C-0161"], indent=1))

> **Check.** The record copies the agent's promise word for word: the courier will call before tomorrow's delivery. But the agent said the order **was delivered** on the 18th of April. The record is faithful to the call, and the promise is wrong. This is a mistake in our own scripts: the agent says this promise even for a delivered order. A check against the order system can catch it (section 6).

## 3. Check 1: the form of each record

Before you trust any field, check the form: every field is there, and the reason and the next step come from the allowed lists. A program that reads the record would break on a missing field or a new word.

In [ ]:
REASONS = ["order_status", "return_refund", "damaged_item", "billing", "product_question", "account_change",
           "cancel_order"]
NEXT_STEPS = ["courier_call", "send_return_label", "send_replacement", "refund_charge", "send_guide",
              "refund_order", "none"]
FIELDS = ["reason", "summary", "promise", "next_step", "recording_notice", "follow_up_needed"]

def shape_problems(answer):
    """What is wrong with the form of one record (an empty list when the form is right)."""
    problems = [f"missing {f}" for f in FIELDS if f not in answer]
    if answer.get("reason") not in REASONS:
        problems.append(f"reason not in the list: {answer.get('reason')!r}")
    if answer.get("next_step") not in NEXT_STEPS:
        problems.append(f"next_step not in the list: {answer.get('next_step')!r}")
    for f in ("recording_notice", "follow_up_needed"):
        if not isinstance(answer.get(f), bool):
            problems.append(f"{f} is not true or false")
    return problems

bad = {cid: shape_problems(a) for cid, a in {**records, **records_asr}.items() if shape_problems(a)}
print(len(bad), "records with a wrong form")
print(shape_problems({"reason": "delivery", "promise": None, "next_step": "courier_call"}))

> **Check.** 0 records with a wrong form in both sets. The last line shows what the check says about a broken record. The LLM was asked for JSON (JSON mode), not held to a schema, so this check is your job.

## 4. Check 2: each field against the labels

Every call has labels: the true reason, the promise the agent said, the next step and the notice. Measure each field on the 50 validation calls. The promise is free text, so it counts as right when it is similar enough to the agent's words (0.6 or more on a scale from 0 to 1).

In [ ]:
import difflib

def similar(a, b, language):
    """0 to 1: how alike two texts are, letter by letter, after the basic normalisation."""
    return difflib.SequenceMatcher(None, cp.normalise(a or "", "basic", language),
                                   cp.normalise(b or "", "basic", language)).ratio()

def field_scores(recs, calls_list):
    """The share of calls where each field of the record agrees with the labels."""
    right = {"reason": 0, "next_step": 0, "follow_up": 0, "notice": 0, "promise": 0}
    for c in calls_list:
        a = recs[c["call_id"]]
        right["reason"] += a.get("reason") == c["reason"]
        right["next_step"] += a.get("next_step") == c["next_step"]
        right["follow_up"] += bool(a.get("follow_up_needed")) == (c["next_step"] != "none")
        right["notice"] += bool(a.get("recording_notice")) == (c["notice"] != "missing")
        right["promise"] += similar(a.get("promise"), c["promise"], c["language"]) >= 0.6
    return {k: round(v / len(calls_list), 3) for k, v in right.items()}

scores = field_scores(records, validation)
print("validation, exact transcripts:", scores)

> **Predict.** The promise scores lower than the other fields. Is the LLM inventing promises?

In [ ]:
for lang in ("en", "fr"):
    group = [c for c in validation if c["language"] == lang]
    ok = sum(similar(records[c["call_id"]].get("promise"), c["promise"], lang) >= 0.6 for c in group)
    print(f"{lang}: promise matches in {ok} of {len(group)} calls")
c = calls["C-0033"]
print()
print("agent said:   ", c["promise"])
print("record says:  ", records["C-0033"]["promise"])
print("similarity:   ", round(similar(records["C-0033"]["promise"], c["promise"], "fr"), 2))

> **Check.** Every English promise matches; only 4 of 16 French promises match. The record for C-0033 says the same thing as the agent, but in English: the model translated it. Letter by letter, the similarity is only 0.43. The check found a real difference (the language), not an invented promise. A person reads the flagged records and decides.

## 5. Your turn: the records from Whisper's transcripts

> **Your turn.** Use `field_scores` on the records that the LLM wrote from Whisper small's transcripts (`records_asr`), for the validation calls. Put the result in `scores_asr`.

In [ ]:
scores_asr = ...

In [ ]:
expect_hash('scores_asr', scores_asr, '76fe65fdb8196866')

> **Check.** The promise matches in more calls (0.9 against 0.76): from Whisper's text, the model kept more French promises in French. One reason changed (0.98): call C-0061, a return that the model called a damaged item. A better score is not always a better record: read what changed.

## 6. Check 3: the promise against the order system

The order system knows the status of each order. The check reads the order number from the transcript (the project's own reader, with the check digit), looks it up, and flags a record that promises a courier call for a delivered order.

In [ ]:
import re

def luhn_ok(number):
    """Larkfield order numbers have 6 digits; the last one is a check digit (Luhn)."""
    total = 0
    for i, ch in enumerate(reversed(number)):
        d = int(ch) * (2 if i % 2 else 1)
        total += d - 9 if d > 9 else d
    return len(number) > 1 and total % 10 == 0

def find_order_number(text, language="en"):
    """(number, problem): the longest run of digits after the normalisation guide, checked."""
    runs = re.findall(r"\d+", cp.normalise(text, "full", language))
    if not runs:
        return None, "no digits"
    best = max(runs, key=len)
    if len(best) != 6:
        return best, "too short" if len(best) < 6 else "too long"
    if not luhn_ok(best):
        return best, "check digit wrong"
    return best, None

def order_check(answer, text, language):
    """'flag' when the record promises a courier call for an order that the system says was delivered."""
    number, problem = find_order_number(text, language)
    if problem or number not in orders:
        return f"no order to check ({problem or 'not in the system'})"
    if orders[number]["status"] == "delivered" and answer.get("next_step") == "courier_call":
        return "flag"
    return "ok"

c = calls["C-0161"]
print(find_order_number(reference(c, "caller")), orders["260026"])
print("exact transcript:", order_check(records["C-0161"], reference(c, "caller"), "en"))

In [ ]:
both = validation + test
flagged = [c["call_id"] for c in both
           if order_check(records[c["call_id"]], reference(c, "caller"), c["language"]) == "flag"]
print("flagged from the exact transcripts:", flagged)

> **Check.** 9 flagged calls on the exact transcripts: every call with this wrong promise in the validation and test sets.

> **Predict.** Run the same check on Whisper small's transcripts. Will it flag the same calls?

## 7. Your turn: the check on recognised transcripts

> **Your turn.** Make the list `flagged_asr`: the IDs of the calls in `both` for which `order_check` says `"flag"`, using the records from Whisper's transcripts (`records_asr`) and Whisper's text (`recognised[call_id]`).

In [ ]:
flagged_asr = ...

In [ ]:
expect_hash('flagged_asr', flagged_asr, 'df0ca4f0caf44e23')

> **Check.** Only 6 of 9. Run the next cell to see why the others were not flagged.

In [ ]:
for cid in flagged:
    c = calls[cid]
    result = order_check(records_asr[cid], recognised[cid], c["language"])
    if result != "flag":
        print(cid, "said:", c["order"], " Whisper small:", find_order_number(recognised[cid], c["language"]), "->", result)

> **Check.** Whisper lost or added a digit, so the number has the wrong length and the check stops. One misheard digit, and a wrong promise stays in the record. Fix: get the order number by keypad (Module 7), and send every record with "no order to check" to a person.

## 8. Failure case: a check that reads the labels

A common mistake when you test a check: you take the order number from the labels, because it is easier.

In [ ]:
cheat = [c["call_id"] for c in both
         if orders.get(c["order"] or "", {}).get("status") == "delivered"
         and records_asr[c["call_id"]].get("next_step") == "courier_call"]
print(len(cheat), "flags when the check reads the order number from the labels")

> **Check.** 9 flags: every wrong promise is caught. In production there are no labels, only Whisper's text, so this test hides the misheard numbers. Test a check on the same input that it will get in production.

## Next

You checked the form, the fields and the promise of each call record, and saw how a transcript error stops a check. The [lesson page](https://learning.nextia-ai.com/courses/speech/m06/summaries-and-follow-ups/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Compliance phrases and keyword spotting](https://learning.nextia-ai.com/courses/speech/m06/compliance-phrases-and-keyword-spotting/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m06/summaries-and-follow-ups/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.